
### Yellow Taxi Trips Cleansed
This notebook cleans the data from the bronze layer for the yellow taxi rides. 

In [0]:
# Import statements
from pyspark.sql.functions import col, when, timestamp_diff
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
# Get the start and end date for the data
two_months_ago_start = date.today().replace(day=1) + relativedelta(months=-2)
one_month_ago_start = date.today().replace(day=1) + relativedelta(months=-1)

In [0]:
# Load data from the bronze layer and filter to include relevant data only
df = spark.read.table("nyctaxi.01_bronze.yellow_taxi_trips_bronze").filter(
    f"tpep_pickup_datetime >= '{two_months_ago_start}' AND tpep_pickup_datetime < '{one_month_ago_start}'"
)

In [0]:
# Further process the data to replace integers with actual values and name columns appropriately
df = df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
        .when(col("VendorID") == 2, "Curb Mobility, LLC")
        .when(col("VendorID") == 6, "Myle Technologies Inc")
        .when(col("VendorID") == 7, "Helix")
        .otherwise("Unknown")
        .alias("vendor"),
    col("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime"),
    timestamp_diff('MINUTE', col("tpep_pickup_datetime"), col("tpep_dropoff_datetime")).alias("trip_duration"),
    col("passenger_count"),
    col("trip_distance"),
    col("RatecodeID"),
    when(col("RatecodeID") == 1, "Standard Rate")
        .when(col("RatecodeID") == 2, "JFK")
        .when(col("RatecodeID") == 3, "Newark")
        .when(col("RatecodeID") == 4, "Nassau or Westchester")
        .when(col("RatecodeID") == 5, "Negotiated Fare")
        .when(col("RatecodeID") == 6, "Group Ride")
        .otherwise("Unknown")
        .alias("rate_type"),
    col("store_and_fwd_flag"),
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    when(col("payment_type") == 0, "Flex Fare Card")
        .when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No charge")
        .when(col("payment_type") == 4, "Dispute")
        .when(col("payment_type") == 6, "Voided trip")
        .otherwise("Unknown")
        .alias("payment_type"),
    col("fare_amount"),
    col("extra"),
    col("mta_tax"),
    col("tolls_amount"),
    col("improvement_surcharge"),
    col("total_amount"),
    col("congestion_surcharge"),
    col("Airport_fee").alias("airport_fee"),
    col("cbd_congestion_fee"),
    col("processed_timestamp")
)

In [0]:
# Save the table
df.write.mode("overwrite").saveAsTable("nyctaxi.02_silver.yellow_taxi_trips_cleansed_silver")